# 4. Automated Response & IOC Matching

When the SOC gets paged at 3 a.m. it's too late to write a runbook. Your response should be **pre-authored** and, where safe, **automated**.

This notebook covers:

1. The response procedures SC-200 tests for each Defender product.
2. **Containment vs eradication vs recovery** — they're not the same thing.
3. **Playbooks**: create, trigger, inspect runs.
4. **Watchlists**: match logs against IOCs (indicators of compromise) exactly like Sentinel's `externaldata()` + `in (…)` pattern.

> **SC-200 mapping**: "Configure automation", "Respond to alerts and incidents across Defender XDR and Sentinel".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [ ]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


## Response procedure cheat sheet (per product)

Memorize these — the exam asks them directly.

### Defender for Office 365 (email)

| Step | Action |
|---|---|
| 1 | Open the email entity page; inspect headers, URLs, attachments (detonation results). |
| 2 | Find all recipients via Advanced Hunting (`EmailEvents`). |
| 3 | **Soft delete** the email from every mailbox (recoverable). |
| 4 | Block sender/domain in the Tenant Allow/Block List. |
| 5 | Check `UrlClickEvents` for anyone who clicked. |
| 6 | If clicked → expand to user-entity investigation. |

### Defender for Endpoint (device)

| Step | Action |
|---|---|
| 1 | Review device timeline and alert process tree. |
| 2 | **Isolate** the device (network containment). |
| 3 | Collect investigation package (forensic bundle). |
| 4 | Run AV scan; quarantine files via `remediate file …` in Live Response. |
| 5 | Release from isolation only after eradication + recovery. |

### Defender for Identity (on-prem AD)

Detects pass-the-hash, pass-the-ticket, golden ticket, DCSync, reconnaissance, lateral movement. Response: disable the account, reset KRBTGT twice if golden ticket, coordinate with AD team.

### Defender for Cloud Apps (SaaS)

Revoke OAuth consent, ban malicious apps, suspend compromised users, require MFA on impossible-travel alerts, block/sanction shadow IT.


## Containment vs eradication vs recovery

| Phase | Goal | Typical actions |
|---|---|---|
| **Containment** | Stop the bleeding **right now**, even if ugly | Isolate device, disable account, revoke tokens, block IP |
| **Eradication** | Remove the attacker's foothold | Delete malware, reset creds, remove persistence, patch the entry point |
| **Recovery** | Return to normal, safely | Re-image hosts, re-enable accounts with MFA, restore data, monitor closely |

A common analyst mistake: calling containment "done" and skipping eradication. The attacker just re-enters.


## Automated response with playbooks

A **playbook** is a set of actions that runs automatically when a rule matches.

Be precise about which product does what — the exam tests the wiring:

| Product | The trigger | The thing that runs |
|---|---|---|
| **Microsoft Sentinel** | **Automation rule** (conditions on incident/alert created or updated) | **Playbook** = an Azure **Logic App** with a Sentinel trigger |
| **Defender XDR** | Detection / incident creation | **AIR** — automated investigation & response — plus **automatic attack disruption**. These are Microsoft-authored, not playbooks you write. |
| **Defender for Endpoint** | Alert | **Automation level** on the device group (Full / Semi / No automated response) governs how far AIR may go without approval |
| **Defender for Cloud** | Recommendation or alert | **Workflow automation** → Logic App |

So: Sentinel is where *you* author response logic; Defender XDR is where *Microsoft's*
response logic runs. When both are onboarded to the unified portal, Sentinel automation
rules can act on Defender XDR incidents too.

Our mini-SIEM lets you POST a playbook and trigger it against an incident:

- `POST /playbooks` — create (⚠️ not upsert; we'll dedupe by name below).
- `GET /playbooks` — list enabled playbooks.
- `POST /playbooks/run/{incident_id}` — run every playbook whose trigger matches the incident's severity and tactic.

### ❌ Bad: one-off manual response every time
Each 3 a.m. page a human runs the same five steps by hand. Inconsistent, slow, exhausting.

### ✅ Best: codified playbooks triggered by severity + tactic


In [ ]:
# Idempotent playbook creation: skip if a playbook with this name already exists
def upsert_playbook(pb):
    existing = {p['name']: p for p in httpx.get(f'{SIEM}/playbooks').json()}
    if pb['name'] in existing:
        return existing[pb['name']]
    return httpx.post(f'{SIEM}/playbooks', json=pb).json()

pb_brute = upsert_playbook({
    'name': 'Contain-BruteForce',
    'trigger_severity': 'High',
    'trigger_tactic': 'CredentialAccess',
    'actions': [
        {'type': 'disable_user', 'note': 'Disable the account in Entra ID'},
        {'type': 'revoke_sessions', 'note': 'Revoke all refresh tokens'},
        {'type': 'require_mfa', 'note': 'Force MFA on next sign-in'},
    ],
})
# NOTE: named after the TACTIC IT TRIGGERS ON. A playbook whose name and trigger
# disagree is a real-world incident-review finding, not a nitpick.
pb_exec = upsert_playbook({
    'name': 'Contain-SuspiciousExecution',
    'trigger_severity': 'High',
    'trigger_tactic': 'Execution',
    'actions': [
        {'type': 'isolate_device', 'note': 'Network-isolate via Defender for Endpoint'},
        {'type': 'collect_package', 'note': 'Collect investigation package'},
        {'type': 'run_av_scan', 'note': 'Full AV scan'},
    ],
})

print('Playbooks registered:')
for p in httpx.get(f'{SIEM}/playbooks').json():
    print(f"  {p['name']:<28} trigger sev={p['trigger_severity']!s:<6} tactic={p['trigger_tactic']}")


In [ ]:
# Trigger matching playbooks on the brute-force incident (if still present)
incidents = httpx.get(f'{SIEM}/incidents').json()
target = next((i for i in incidents if 'Brute force' in i['title']), None)
if target:
    result = httpx.post(f"{SIEM}/playbooks/run/{target['id']}").json()
    print(f"Ran playbooks on {target['id']}:")
    for r in result['playbooks_executed']:
        print(f"  ▶ {r['playbook']}  (run {r['run_id']})")
        for action in r['actions']:
            desc = action.get('note') or action.get('description') or ''
            print(f"      - {action['type']:<16} {desc}")
else:
    print('No brute-force incident present (notebook 3 may have closed all of them).')


## Watchlists and IOC matching

A **watchlist** is a named list of indicators (IPs, domains, file hashes, VIP users). You maintain it in one place and match every log against it.

In Sentinel:

```
let bad_ips = _GetWatchlist('KnownBadIPs') | project IPAddress;
AzureFirewall | where DestinationIP in (bad_ips)
```

Our mini-SIEM exposes the same shape: `POST /watchlists` (upsert) and `POST /watchlists/match`.


In [ ]:
# 1. Upsert a watchlist of known-bad IPs (upsert = safe to re-run)
httpx.post(f'{SIEM}/watchlists', json={
    'name': 'KnownBadIPs',
    'description': 'C2 infrastructure from threat intel feed',
    'items': ['185.220.101.42', '45.33.32.156', '198.51.100.99'],
})

# 2. Match firewall logs against the watchlist
match = httpx.post(f'{SIEM}/watchlists/match', json={
    'watchlist': 'KnownBadIPs',
    'table_name': 'AzureFirewall',
    'field': 'DestinationIP',
    'time_range_minutes': 1440,
    'limit': 50,
}).json()

print(f"Matches for KnownBadIPs: {match['match_count']}")
for m in match['matches'][:5]:
    print(f"  [{m['timestamp'][:19]}] {m['SourceIP']} → {m['DestinationIP']}:{m['DestinationPort']} ({m['Action']})")


In [ ]:
# IOC matching also works on user watchlists — e.g. VIP accounts to watch extra-closely
httpx.post(f'{SIEM}/watchlists', json={
    'name': 'VIPUsers',
    'description': 'Executives & privileged accounts — scrutinize every sign-in',
    'items': ['alice@contoso.com'],
})

vip_hits = httpx.post(f'{SIEM}/watchlists/match', json={
    'watchlist': 'VIPUsers',
    'table_name': 'SigninLogs',
    'field': 'UserPrincipalName',
    'time_range_minutes': 1440,
    'limit': 200,
}).json()

failed_vip = [m for m in vip_hits['matches'] if m['ResultType'] == 'Failure']
print(f"VIP sign-ins in last 24h: {vip_hits['match_count']}  (failures: {len(failed_vip)})")


## When to trust automation (and when not to)

Automation is a force multiplier, but it can also do damage at machine speed. SC-200 expects you to reason about **blast radius of the automation itself**.

| Action | Safe to fully automate? | Why |
|---|---|---|
| **Isolate a device** | ✅ Usually | Reversible in one click; stops exfil immediately. |
| **Disable a user** | ⚠️ With care | Can page the whole exec team at 3 a.m. if wrong. Scope to non-VIP accounts or require analyst approval. |
| **Delete email from mailboxes** | ✅ Soft delete only | Soft delete is recoverable; **never** auto-hard-delete. |
| **Reset a password** | ⚠️ With care | Can lock out a real user mid-flight. Combine with token revocation + MFA. |
| **Re-image a device** | ❌ Never auto | Data-destructive; requires human sign-off. |

Defender XDR's **automatic attack disruption** follows exactly these rules: it disables users, contains devices, and blocks OAuth apps — but it does not delete user data.


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| `POST /playbooks` | Sentinel → Automation rule + Logic App |
| `POST /playbooks/run/{incident}` | XDR automation rule firing on an incident |
| `POST /watchlists` | Sentinel → Watchlists |
| `POST /watchlists/match` | KQL `in (_GetWatchlist(…))` |

### Exam tips

- Know the **containment / eradication / recovery** phases and give an action for each.
- Know **soft vs hard delete** for email; only soft delete is auto-safe.
- **Playbook triggers** match on severity, tactic, or rule — not on free-text.
- **Watchlists** are the right tool for maintained lists of IOCs or VIPs; do not hard-code them in every rule.

🎉 You've completed Lab 2. In Lab 3 you'll proactively *hunt* with the same SIEM.


---
## ✅ Self-check

1. In Sentinel, what fires a playbook, and what *is* a playbook technically?
2. Name one response action that is always safe to fully automate and one that must never be.
3. A user's mailbox received a malicious email that went to 400 recipients. What is the
   response action, and what is the one thing you must not automate?
4. You isolated the device and removed the malware. Are you done? Which phase is missing?
5. Your IOC list of C2 IPs is referenced by nine analytics rules. Where should it live and why?
6. What does Defender XDR's automatic attack disruption do, and what does it deliberately
   *not* do?

In [ ]:
answers = """
1. An AUTOMATION RULE fires it (on incident created/updated, or on alert, with
   conditions on severity/title/tactic/tag/owner). The playbook itself is an AZURE
   LOGIC APP with a Microsoft Sentinel trigger -- so it can call any connector:
   Entra ID, Defender for Endpoint, Teams, ServiceNow, a firewall API.

2. Always safe: ISOLATING A DEVICE (reversible in one click, stops exfiltration
   immediately) or SOFT-DELETING a malicious email (recoverable).
   Never: RE-IMAGING / WIPING a device -- it is data-destructive and irreversible;
   it requires human sign-off. Hard-deleting mail is the email equivalent.

3. Soft delete the message from all 400 mailboxes (Defender for Office 365
   'Take action' / Threat Explorer), block the sender and URL in the Tenant
   Allow/Block List, then check UrlClickEvents for who clicked. Do NOT automate a
   HARD delete -- once purged the evidence is gone and a mistake is unrecoverable.

4. NO -- you did containment (isolate) and eradication (remove malware). RECOVERY is
   missing: re-image or verify the host, restore data, re-enable the account with MFA,
   remove the isolation, and monitor closely for re-entry. Skipping recovery leaves the
   business impaired; skipping eradication lets the attacker back in.

5. In a WATCHLIST (Sentinel) or as threat intelligence indicators. One place to update,
   nine rules that reference it with _GetWatchlist(...) / the TI tables. Hard-coding
   the list in nine queries guarantees they drift out of sync, and every IOC update
   becomes nine rule edits and nine change reviews.

6. It automatically CONTAINS an in-progress attack at machine speed: disables the
   compromised user account, contains the device, and disables/blocks malicious OAuth
   apps -- based on high-confidence XDR signals. It deliberately does NOT delete user
   data, wipe devices, or take irreversible destructive actions, and every action it
   takes is reversible from the portal.
"""
print(answers)